# SynFlow Top-5: L2 Evaluation (TPU, 50 epochs)

Fair comparison: SynFlow top-5 vs HBO top-5 vs Random top-5

SynFlow top-5 (from local CPU computation):
1. W=96 D=6 BN NoSkip  (score=3.80e+02)
2. W=96 D=5 BN NoSkip  (score=3.58e+02)
3. W=96 D=3 BN NoSkip  (score=3.38e+02)
4. W=64 D=6 BN NoSkip  (score=3.03e+02)
5. W=64 D=5 BN NoSkip  (score=2.79e+02)

In [1]:
import sys
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

# TPU setup
import torch_xla.core.xla_model as xm
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

DEVICE = xm.xla_device()
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


/tmp/ipykernel_73/2281108365.py:17: DeprecationWarning: Use torch_xla.device instead
  DEVICE = xm.xla_device()
E0000 00:00:1776729500.847050      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238


Device: xla:0
PyTorch version: 2.8.0+cpu


In [2]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url} --branch develop
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Cloned develop branch")

/kaggle/working


Cloning into 'ThermoRG-NN'...


remote: Enumerating objects: 1630, done.
remote: Counting objects: 100% (84/84), done.


remote: Compressing objects: 100% (79/79), done.


remote: Total 1630 (delta 7), reused 7 (delta 5), pack-reused 1546 (from 3)
Receiving objects: 100% (1630/1630), 2.81 MiB | 17.33 MiB/s, done.
Resolving deltas: 100% (913/913), done.


/kaggle/working/ThermoRG-NN


Cloned develop branch


In [3]:
# Data loading — same protocol as HBO_revised notebook
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

full_train = torchvision.datasets.CIFAR10('/tmp/data', train=True, download=True, transform=transform_train)
test_ds = torchvision.datasets.CIFAR10('/tmp/data', train=False, download=True, transform=transform_test)

# Split into train/val (same seed as HBO_revised)
np.random.seed(42)
indices = np.random.permutation(len(full_train))
val_size = 5000
val_idx, train_idx = indices[:val_size], indices[val_size:]
train_ds = Subset(full_train, train_idx)
val_ds = Subset(full_train, val_idx)

BATCH_SIZE = 256

# 终极修复：彻底关闭多进程 (num_workers=0) 并防止不完整批次导致 TPU 图重编译 (drop_last=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=True)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

  0%|          | 0.00/170M [00:00<?, ?B/s]

  0%|          | 426k/170M [00:00<00:42, 4.03MB/s]

  3%|▎         | 5.80M/170M [00:00<00:05, 32.5MB/s]

 10%|▉         | 16.3M/170M [00:00<00:02, 65.0MB/s]

 15%|█▌        | 26.3M/170M [00:00<00:01, 78.8MB/s]

...

 89%|████████▊ | 151M/170M [00:01<00:00, 101MB/s] 

 95%|█████████▍| 161M/170M [00:01<00:00, 97.9MB/s]

100%|██████████| 170M/170M [00:01<00:00, 91.5MB/s]

/usr/local/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 45000, Val: 5000, Test: 10000


In [4]:
# SynFlow top-5 configs (from local CPU computation, same search space as HBO)
# These are the highest-SynFlow-score architectures in the search space
SYNFLOW_TOP5 = [
    {'width': 96, 'depth': 6, 'norm': 'batchnorm', 'skip': False},  # rank 1  score=3.80e+02
    {'width': 96, 'depth': 5, 'norm': 'batchnorm', 'skip': False},  # rank 2  score=3.58e+02
    {'width': 96, 'depth': 3, 'norm': 'batchnorm', 'skip': False},  # rank 3  score=3.38e+02
    {'width': 64, 'depth': 6, 'norm': 'batchnorm', 'skip': False},  # rank 4  score=3.03e+02
    {'width': 64, 'depth': 5, 'norm': 'batchnorm', 'skip': False},  # rank 5  score=2.79e+02
]

EPOCHS_L1 = 10   # L1 screening
EPOCHS_L2 = 50   # L2 final evaluation
LR = 0.05
SEEDS = [42, 123]
CHECKPOINT_EVERY = 10
LOG_EVERY = 5

In [5]:
def build_model(width, depth, norm_type, skip=False):
    """Plain ConvNet matching HBO_revised architecture definition."""
    layers = []
    c = 3
    for i in range(depth):
        layers.append(nn.Conv2d(c, width, 3, padding=1))
        
        if norm_type == 'batchnorm':
            layers.append(nn.BatchNorm2d(width))
        elif norm_type == 'layernorm':
            layers.append(nn.LayerNorm([width, 32, 32]))
            
        # 终极修复：加入论文中明确规定的 GELU 激活函数！
        layers.append(nn.GELU())
        
        c = width
        
    if skip:
        layers.append(nn.Identity())  # placeholder for skip
        
    layers.extend([
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(c, 10)
    ])
    return nn.Sequential(*layers)

In [6]:
import torch_xla.distributed.parallel_loader as pl
import torch
import torch.nn.functional as F

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    # 使用 XLA 官方提供的异步数据加载器，解放 CPU
    mp_loader = pl.MpDeviceLoader(loader, DEVICE)
    
    # 将累加器全部定义在 TPU 上，过程完全 0 通信
    total_loss = torch.tensor(0.0, device=DEVICE)
    total_samples = torch.tensor(0.0, device=DEVICE)
    
    for x, y in mp_loader:
        optimizer.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        xm.optimizer_step(optimizer)
        
        total_loss += loss.detach() * x.size(0)
        total_samples += x.size(0)
        
        # 强制结算当前批次，防止计算图无限庞大撑爆内存
        xm.mark_step() 
        
    scheduler.step()
    # 整个 epoch 结束时，统一拉回 CPU 结算一次
    return (total_loss / total_samples).item()


def evaluate(model, loader):
    model.eval()
    mp_loader = pl.MpDeviceLoader(loader, DEVICE)
    
    total_loss = torch.tensor(0.0, device=DEVICE)
    correct = torch.tensor(0.0, device=DEVICE)
    total_samples = torch.tensor(0.0, device=DEVICE)
    
    with torch.no_grad():
        for x, y in mp_loader:
            out = model(x)
            loss = F.cross_entropy(out, y)
            pred = out.argmax(dim=1)
            
            total_loss += loss.detach() * x.size(0)
            correct += (pred == y).sum()
            total_samples += x.size(0)
            
            xm.mark_step() 
            
    return (total_loss / total_samples).item(), (correct / total_samples).item()

In [7]:
import gc
import time

def train_model(cfg, epochs, seed=42, log_every=5):
    """Train one config, return full epoch-by-epoch history."""
    torch.manual_seed(seed)
    model = build_model(cfg['width'], cfg['depth'], cfg['norm'], cfg['skip'])
    model = model.to(DEVICE)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = []
    best_val_loss = float('inf')
    best_val_acc = 0.0
    start = time.time()
    
    for ep in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler)
        val_loss, val_acc = evaluate(model, val_loader)
        
        # Track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
        
        # Log every N epochs
        if (ep + 1) % log_every == 0 or ep == 0:
            elapsed = time.time() - start
            print(f"    ep={ep+1:3d}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}  ({elapsed:.0f}s)", flush=True)
        
        # Checkpoint
        if (ep + 1) % 10 == 0:
            xm.save(model.state_dict(), f"/tmp/ckpt_W{cfg['width']}_D{cfg['depth']}_ep{ep+1}_seed{seed}.pt")
        
        history.append({'epoch': ep+1, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc})
    
    test_loss, test_acc = evaluate(model, test_loader)
    
    # 彻底抹除驻留在 TPU 内存中的计算图与缓存，防止跨 Config 搜索时 OOM
    del model
    del optimizer
    del scheduler
    gc.collect()
    xm.mark_step() 
    
    return {
        'history': history,
        'best_val_loss': best_val_loss,
        'best_val_acc': best_val_acc,
        'test_loss': test_loss,
        'test_acc': test_acc
    }

In [8]:
# ── Phase 1: L1 (10 epochs) ─────────────────────────────────────────────
print("=" * 70)
print("Phase 1: L1 (10 epochs) — SynFlow top-5")
print("=" * 70)

sys.path.insert(0, '/home/node/.openclaw/workspace/github_staging/ThermoRG-NN')
from thermorg.topology_calculator import compute_J_topo

l1_results = []
for i, cfg in enumerate(SYNFLOW_TOP5):
    cfg_label = f"W={cfg['width']} D={cfg['depth']} {cfg['norm']} skip={cfg['skip']}"
    print(f"\n[{i+1}/5] {cfg_label}")
    
    # Compute J_topo
    model_for_J = build_model(cfg['width'], cfg['depth'], cfg['norm'], cfg['skip'])
    J, _ = compute_J_topo(model_for_J)
    
    l1_losses = []
    for seed in SEEDS:
        t0 = time.time()
        result = train_model(cfg, EPOCHS_L1, seed, log_every=5)
        elapsed = time.time() - t0
        l1_losses.append(result['best_val_loss'])
        print(f"  seed={seed}: L1_best_val={result['best_val_loss']:.4f}  ({elapsed:.0f}s)")
    
    l1_results.append({
        'cfg': cfg,
        'J_topo': J,
        'l1_losses': l1_losses,
        'l1_mean': np.mean(l1_losses)
    })
    print(f"  → L1 mean: {np.mean(l1_losses):.4f}  J_topo={J:.4f}")

Phase 1: L1 (10 epochs) — SynFlow top-5

[1/5] W=96 D=6 batchnorm skip=False


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


/tmp/ipykernel_73/195971708.py:25: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


/tmp/ipykernel_73/195971708.py:50: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


    ep=  1/10  train=1.6144  val=1.4014  acc=0.4799  (53s)


...

  seed=42: L1_best_val=0.7097  (200s)


    ep=  1/10  train=1.6732  val=1.4563  acc=0.4603  (13s)


    ep=  5/10  train=0.9839  val=1.0952  acc=0.6067  (64s)


    ep= 10/10  train=0.7180  val=0.7250  acc=0.7510  (129s)


  seed=123: L1_best_val=0.7250  (130s)
  → L1 mean: 0.7174  J_topo=0.5433


In [9]:
# ── Phase 2: L2 (50 epochs) ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("Phase 2: L2 (50 epochs) — SynFlow top-5")
print("=" * 70)

l2_results = []
for i, cfg in enumerate(SYNFLOW_TOP5):
    cfg_label = f"W={cfg['width']} D={cfg['depth']} {cfg['norm']} skip={cfg['skip']}"
    print(f"\n[{i+1}/5] {cfg_label}")
    
    l2_losses = []
    l2_accs = []
    for seed in SEEDS:
        t0 = time.time()
        result = train_model(cfg, EPOCHS_L2, seed, log_every=10)
        elapsed = time.time() - t0
        l2_losses.append(result['best_val_loss'])
        l2_accs.append(result['test_acc'])
        print(f"  seed={seed}: L2_val={result['best_val_loss']:.4f}  test_acc={result['test_acc']:.4f}  ({elapsed:.0f}s)", flush=True)
    
    l2_mean = np.mean(l2_losses)
    l2_std = np.std(l2_losses)
    l2_acc_mean = np.mean(l2_accs)
    l2_results.append({
        'cfg': cfg,
        'l2_losses': l2_losses,
        'l2_mean': l2_mean,
        'l2_std': l2_std,
        'l2_acc_mean': l2_acc_mean
    })
    print(f"  → L2 mean: {l2_mean:.4f} ± {l2_std:.4f}  test_acc: {l2_acc_mean:.4f}")


Phase 2: L2 (50 epochs) — SynFlow top-5

[1/5] W=96 D=6 batchnorm skip=False


/tmp/ipykernel_73/195971708.py:25: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


/tmp/ipykernel_73/195971708.py:50: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()


    ep=  1/50  train=1.6144  val=1.4014  acc=0.4799  (13s)


    ep= 10/50  train=0.6404  val=0.7251  acc=0.7496  (128s)


...

    ep= 30/50  train=0.4147  val=0.5918  acc=0.7969  (384s)


    ep= 40/50  train=0.3067  val=0.4792  acc=0.8353  (512s)


    ep= 50/50  train=0.2617  val=0.4538  acc=0.8448  (639s)


  seed=123: L2_val=0.4484  test_acc=0.8405  (641s)


  → L2 mean: 0.4399 ± 0.0085  test_acc: 0.8398


In [10]:
# ── Summary ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL RESULTS: SynFlow Top-5 — L2 (50 epochs)")
print("=" * 70)
print(f"{'Rank':<5} {'Config':<25} {'J_topo':>8} {'L2 Loss':>10} {'Test Acc':>10}")
print("-" * 70)

# Sort by L2 mean
sorted_results = sorted(l2_results, key=lambda x: x['l2_mean'])
for rank, r in enumerate(sorted_results, 1):
    cfg = r['cfg']
    label = f"W={cfg['width']} D={cfg['depth']} BN NS"
    # Get J_topo from L1 results (same cfg)
    j_topo = next(x['J_topo'] for x in l1_results if x['cfg'] == cfg)
    print(f"{rank:<5} {label:<25} {j_topo:>8.4f} {r['l2_mean']:>10.4f} {r['l2_acc_mean']:>10.4f}")

print("\n" + "-" * 70)
print("Comparison with HBO and Random (50-epoch L2 from HBO_revised run):")
print("-" * 70)
print(f"{'Method':<15} {'Best Config':<25} {'Best Loss':>10} {'Best Acc':>10}")
print("-" * 70)
print(f"{'SynFlow':<15} {'W=96 D=6 BN NoSkip':<25} {sorted_results[0]['l2_mean']:>10.4f} {sorted_results[0]['l2_acc_mean']:>10.4f}")
print(f"{'HBO':<15} {'W=96 D=6 BN NoSkip':<25} {0.3770:>10.4f} {0.8744:>10.4f}")
print(f"{'Random':<15} {'W=64 D=6 BN NoSkip':<25} {0.4270:>10.4f} {0.8515:>10.4f}")


FINAL RESULTS: SynFlow Top-5 — L2 (50 epochs)
Rank  Config                      J_topo    L2 Loss   Test Acc
----------------------------------------------------------------------
1     W=96 D=6 BN NS              0.5425     0.3527     0.8724
2     W=64 D=6 BN NS              0.5880     0.3963     0.8587
3     W=96 D=5 BN NS              0.4766     0.3993     0.8607
4     W=64 D=5 BN NS              0.5433     0.4399     0.8398
5     W=96 D=3 BN NS              0.2829     0.6670     0.7631

----------------------------------------------------------------------
Comparison with HBO and Random (50-epoch L2 from HBO_revised run):
----------------------------------------------------------------------
Method          Best Config                Best Loss   Best Acc
----------------------------------------------------------------------
SynFlow         W=96 D=6 BN NoSkip            0.3527     0.8724
HBO             W=96 D=6 BN NoSkip            0.3770     0.8744
Random          W=64 D=6 BN NoS

In [11]:
# Save results for figure generation
output = {
    'experiment': 'synflow_l2_tpu',
    'epochs_l1': EPOCHS_L1,
    'epochs_l2': EPOCHS_L2,
    'seeds': SEEDS,
    'synflow_top5': [
        {
            'cfg': r['cfg'],
            'J_topo': next(x['J_topo'] for x in l1_results if x['cfg'] == r['cfg']),
            'l1_mean': next(x['l1_mean'] for x in l1_results if x['cfg'] == r['cfg']),
            'l2_mean': r['l2_mean'],
            'l2_std': r['l2_std'],
            'l2_acc_mean': r['l2_acc_mean']
        }
        for r in l2_results
    ],
    'hbo_reference': {
        'best_loss': 0.3770,
        'best_acc': 0.8744,
        'best_config': 'W=96 D=6 BN NoSkip'
    },
    'random_reference': {
        'best_loss': 0.4270,
        'best_acc': 0.8515,
        'best_config': 'W=64 D=6 BN NoSkip'
    }
}

with open('/tmp/synflow_l2_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print("Results saved to /tmp/synflow_l2_results.json")

# Print JSON for direct copy
print("\nJSON for paper:\n")
print(json.dumps(output, indent=2))

Results saved to /tmp/synflow_l2_results.json

JSON for paper:

{
  "experiment": "synflow_l2_tpu",
  "epochs_l1": 10,
  "epochs_l2": 50,
  "seeds": [
    42,
    123
  ],
  "synflow_top5": [
    {
      "cfg": {
        "width": 96,
        "depth": 6,
        "norm": "batchnorm",
        "skip": false
      },
      "J_topo": 0.5424963193604435,
      "l1_mean": 0.6020120680332184,
      "l2_mean": 0.352693110704422,
      "l2_std": 0.006333440542221069,
      "l2_acc_mean": 0.8723958432674408
    },
    {
      "cfg": {
        "width": 96,
        "depth": 5,
        "norm": "batchnorm",
        "skip": false
      },
      "J_topo": 0.476557040622864,
      "l1_mean": 0.6845841705799103,
      "l2_mean": 0.39931850135326385,
      "l2_std": 0.0025121718645095825,
      "l2_acc_mean": 0.8606770932674408
    },
    {
      "cfg": {
        "width": 96,
        "depth": 3,
        "norm": "batchnorm",
        "skip": false
      },
      "J_topo": 0.2828514958406031,
      "l1_mean":